In [2]:

import re
from collections import defaultdict
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

# Le carte Iran Economico hanno tutte risorse_iran:(v)=>v-1
# Questo è CORRETTO semanticamente: spendono risorse Iran
# Manca invece il lato positivo: carte che AUMENTANO risorse_iran
# 
# Guardiamo le carte Iran che NON sono Economico e aggiungiamo risorse_iran:(v)=>v+1
# Carte Iran tipo Diplomatico (accordi commerciali, diplomazia aumenta risorse)

card_re = re.compile(
    r"\{\s*card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)',\s*description:'([^']*)'",
    re.DOTALL
)
matches = list(card_re.finditer(src))

def add_effect_to_card(src, cid, track, delta_expr, matches):
    card_search = f"card_id:'{cid}'"
    pos = src.find(card_search)
    if pos == -1:
        return src, False
    eff_m = re.search(r'(effects:\{)([^}]+)(\})', src[pos:pos+900])
    if not eff_m:
        return src, False
    eff_str = eff_m.group(2)
    if track + ':' in eff_str:
        return src, False
    new_eff = eff_m.group(1) + eff_m.group(2) + f', {track}:(v)=>{delta_expr}' + eff_m.group(3)
    abs_pos = pos + eff_m.start()
    new_src = src[:abs_pos] + new_eff + src[abs_pos + len(eff_m.group(0)):]
    return new_src, True

new_src = src
added = 0

# risorse_iran positivo: carte Iran Diplomatico (diplomazia → risorse)
for idx, m in enumerate(matches):
    if added >= 8: break
    cid, cname, faz, ctype, op, deck, desc = m.groups()
    if faz != 'Iran' or ctype != 'Diplomatico': continue
    new_src, ok = add_effect_to_card(new_src, cid, 'risorse_iran', 'v+1', matches)
    if ok: added += 1

print(f"risorse_iran pos aggiunti: {added}")

added2 = 0
# risorse_cina positivo: carte Cina Diplomatico
for idx, m in enumerate(matches):
    if added2 >= 8: break
    cid, cname, faz, ctype, op, deck, desc = m.groups()
    if faz != 'Cina' or ctype != 'Diplomatico': continue
    new_src, ok = add_effect_to_card(new_src, cid, 'risorse_cina', 'v+1', matches)
    if ok: added2 += 1

print(f"risorse_cina pos aggiunti: {added2}")

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts', 'w') as f:
    f.write(new_src)

print("File scritto.")


risorse_iran pos aggiunti: 8
risorse_cina pos aggiunti: 8
File scritto.
